<a href="https://colab.research.google.com/github/haribharadwaj/notebooks/blob/main/uPNC/ACC_ITD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using the EEG "change complex" as a correlate of ITD sensitivity

In this notebook we analyze EEG recordings of the **acoustic change complex (ACC)** evoked by **interaural time difference (ITD) jumps**. The ITD is the small difference in the arrival time of a sound at the two ears, and it is one of the main cues the auditory system uses to localize sound and to separate sources in space. Here we present an ongoing sound whose ITD suddenly *jumps* part way through, and we measure the cortical response to that change.

## What you'll do

You will run the **analysis cell twice** — once for a **small** ITD jump and once for a **large** ITD jump — and **plot the evoked response after each run**, so that both responses appear on the *same* set of axes for comparison.

The events in the recording are coded as:

| Event code | ITD jump size | Direction of change          |
|:----------:|:-------------:|:-----------------------------|
| 1          | Small         | left-leading → right-leading |
| 5          | Small         | right-leading → left-leading |
| 3          | Large         | left-leading → right-leading |
| 7          | Large         | right-leading → left-leading |

So the **small** ITD condition uses events `[1, 5]` and the **large** ITD condition uses events `[3, 7]`.

## What to expect in the results

The ITD change happens at about **t = 1 s** in each epoch.

- **Small ITD jump:** a *weak* cortical response. You essentially see only a **target-detection / change-detection response that begins around 300 ms after the change** (i.e. around t ≈ 1.3 s), with little or no early sensory response.
- **Large ITD jump:** a *strong* cortical response. You see a clear **sensory N1–P2 complex**, with the **N1 around t ≈ 1.1 s** and the **P2 around t ≈ 1.2 s**, in addition to the later change-detection activity.

Comparing the two tells you how robustly cortex registers a spatial change as a function of how large that change is — a neural correlate of ITD sensitivity.

## Tools and references

We use **MNE-Python**, an open-source package for analyzing electrophysiology (EEG/MEG) data. The official package can be found below, but we will use Hari's fork of it from GitHub when we install in the next cell.

- MNE-Python on GitHub: https://github.com/mne-tools/mne-python
- MNE-Python documentation: https://mne.tools/

The acoustic change complex paradigm used here is described in:

> Borjigin, A., Hustedt-Mai, A. R., & Bharadwaj, H. M. (2022). Individualized Assays of Temporal Coding in the Ascending Human Auditory System. eNeuro, 9(2), ENEURO.0378-21.2022. https://doi.org/10.1523/ENEURO.0378-21.2022

Blink (eye-movement) artifacts are removed using the **Signal-Space Projection (SSP)** method:

> Uusitalo, M. A., & Ilmoniemi, R. J. (1997). *Signal-space projection method for separating MEG or EEG into components.* Medical and Biological Engineering and Computing, 35(2), 135–140.


## Setup for Google Colab

Because we are running this notebook in **Google Colab**, run the next two cells **first**, before anything else.

1. The first cell installs the specific fork of **MNE-Python** used for this exercise (installed directly from GitHub, so you get the correct version).
2. The second cell downloads the EEG data files into a folder called `eegdata/` on your Colab machine.

These steps only need to be run **once per Colab session**. If your session disconnects or you start a fresh one, run them again. After both cells finish, set `fpath = '/content/eegdata/'` in the Analysis cell so the analysis can find the downloaded files.

In [ ]:
# Install the course fork of MNE-Python (run once per Colab session)
!pip -q install git+https://github.com/haribharadwaj/mne-python

### Download the EEG data

This cell creates the `eegdata/` folder and downloads the three `.bdf` recordings into it from Dropbox. After this cell completes, you should see three `.bdf` files listed.

In [ ]:
# Download the EEG data into /content/eegdata/ (run once per Colab session)
!mkdir -p eegdata
!wget -q --show-progress -O eegdata/EEG_file1.bdf  "https://www.dropbox.com/scl/fi/te8xeex75rq800jjqmtsi/S197_ITD.bdf?rlkey=u54ekyck0i2ln9vxaxwzcvb5o&st=ii995mt5&dl=1"
!wget -q --show-progress -O eegdata/EEG_file2.bdf  "https://www.dropbox.com/scl/fi/2w8ospda6cykd0j9buq99/S197_ITD-001.bdf?rlkey=u0bay00rs4v45vm9jrduxm2yv&st=g6zvlqaz&dl=1"
!wget -q --show-progress -O eegdata/EEG_file3.bdf  "https://www.dropbox.com/scl/fi/yrk79opse21ptilu3mbn0/S197_ITD-002.bdf?rlkey=p9qn6h4zvzx8jku71enxx92q5&st=novjmukw&dl=1"

# Quick check that the files arrived
!ls -lh eegdata/

## Imports

We load the libraries used throughout the notebook:

- **`mne`** — the main EEG/MEG analysis package (reading the raw data, finding events, epoching, averaging, plotting).
- **`compute_proj_epochs`** — the MNE routine that builds the **SSP projectors** we use to suppress blink artifacts (see Uusitalo & Ilmoniemi, 1997).
- **`fnmatch`, `os`, `sys`** — standard Python utilities for finding the data files on disk.
- **`matplotlib.pyplot`** — for plotting the evoked responses.
- **`scipy.io`** — for any MATLAB-format file I/O.

Just run this cell; it produces no output.


In [ ]:
import mne
import fnmatch
import os
import sys
from mne.preprocessing.ssp import compute_proj_epochs
import matplotlib.pyplot as plt
from scipy import io

## Some useful function definitions

This cell defines two helper functions and produces no visible output — just run it so the functions are available later.

- **`find_blinks(...)`** detects eye-blink events directly from a frontal EEG channel (default `A1`). It band-pass filters that channel to emphasize the slow, large blink waveform, finds the peaks, and keeps only the peaks whose size looks like a typical blink. It returns the blink times in MNE's event format. The analysis cell feeds these detected blinks into SSP so the blink component can be projected out of the data.
- **`peak_finder(...)`** is a robust, noise-tolerant peak-detection routine used by `find_blinks`. (It is a Python port of a well-known MATLAB peak finder.)

You do **not** need to read these in detail to do the exercise — they are infrastructure for the blink-removal step.


In [ ]:
import numpy as np

from math import ceil
from mne import pick_channels
from mne.filter import filter_data

def find_blinks(raw, event_id=998, thresh=100e-6, l_freq=0.5, h_freq=10,
                filter_length='auto', ch_name=['A1', ], tstart=0.,
                l_trans_bandwidth=0.15):

    """Utility function to detect blink events from specified channel.

    Parameters
    ----------
    raw : instance of Raw
        The raw data.
    event_id : int
        The index to assign to found events.
    low_pass : float
        Low pass frequency.
    high_pass : float
        High pass frequency.
    filter_length : str | int | None
        Number of taps to use for filtering.
    ch_name: list | None
        If not None, use specified channel(s) for EOG
    tstart : float
        Start detection after tstart seconds.

    Returns
    -------
    eog_events : array
        Events in MNE  format, i.e., N x 3 array
    """

    sampling_rate = raw.info['sfreq']
    first_samp = raw.first_samp

    ch_eog = pick_channels(raw.ch_names, include=ch_name)

    if len(ch_eog) == 0:
        raise ValueError('%s not in channel list' % ch_name)

    eog, _ = raw[ch_eog, :]
    filteog = filter_data(eog, sampling_rate, l_freq, h_freq,
                          filter_length=filter_length,
                          l_trans_bandwidth=l_trans_bandwidth)

    eog_events, blinkvals = peak_finder(filteog.squeeze(), thresh=thresh)
    eog_events_neg, blinkvals_neg = peak_finder(filteog.squeeze(),
                                                thresh=thresh, extrema=-1)

    # Discarding blinks that don't look like other blinks, electing polarity
    nominal_blink = np.median(np.abs(blinkvals))
    nominal_blink_neg = np.median(np.abs(blinkvals_neg))

    if nominal_blink_neg > nominal_blink:
        blinkvals = blinkvals_neg
        nominal_blink = nominal_blink_neg
        eog_events = eog_events_neg

    eog_events = eog_events[np.logical_and(np.abs(blinkvals) < 2*nominal_blink,
                                           np.abs(blinkvals) >
                                           0.5*nominal_blink)]

    # Discarding blinks detected before tstart seconds
    eog_events = eog_events[eog_events > raw.time_as_index(tstart)]
    eog_events += first_samp
    n_events = len(eog_events)
    print(f'Number of EOG events detected : {n_events}')
    eog_events = np.c_[eog_events, np.zeros(n_events),
                       event_id * np.ones(n_events)]

    return np.int64(eog_events)



def peak_finder(x0, thresh=None, extrema=1):
    """Noise tolerant fast peak finding algorithm

    Parameters
    ----------
    x0 : 1d array
        A real vector from the maxima will be found (required).
    thresh : float
        The amount above surrounding data for a peak to be
        identified (default = (max(x0)-min(x0))/4). Larger values mean
        the algorithm is more selective in finding peaks.
    extrema : {-1, 1}
        1 if maxima are desired, -1 if minima are desired
        (default = maxima, 1).

    Returns
    -------
    peak_loc : array
        The indices of the identified peaks in x0
    peak_mag : array
        The magnitude of the identified peaks

    Note
    ----
    If repeated values are found the first is identified as the peak.
    Conversion from initial Matlab code from:
    Nathanael C. Yoder (ncyoder@purdue.edu)

    Example
    -------
    t = 0:.0001:10;
    x = 12*sin(10*2*pi*t)-3*sin(.1*2*pi*t)+randn(1,numel(t));
    x(1250:1255) = max(x);
    peak_finder(x)
    """

    x0 = np.asanyarray(x0)

    if x0.ndim >= 2:
        raise ValueError('The input data must be a 1D vector')

    s = x0.size

    if thresh is None:
        thresh = (np.max(x0) - np.min(x0)) / 4

    assert extrema in [-1, 1]

    if extrema == -1:
        x0 = extrema * x0  # Make it so we are finding maxima regardless

    dx0 = np.diff(x0)  # Find derivative
    # This is so we find the first of repeated values
    dx0[dx0 == 0] = -np.finfo(float).eps
    # Find where the derivative changes sign
    ind = np.where(dx0[:-1:] * dx0[1::] < 0)[0] + 1

    # Include endpoints in potential peaks and valleys
    x = np.concatenate((x0[:1], x0[ind], x0[-1:]))
    ind = np.concatenate(([0], ind, [s - 1]))

    #  x only has the peaks, valleys, and endpoints
    length = x.size
    min_mag = np.min(x)

    if length > 2:  # Function with peaks and valleys

        # Set initial parameters for loop
        temp_mag = min_mag
        found_peak = False
        left_min = min_mag

        # Deal with first point a little differently since tacked it on
        # Calculate the sign of the derivative since we taked the first point
        # on it does not necessarily alternate like the rest.
        signDx = np.sign(np.diff(x[:3]))
        if signDx[0] <= 0:  # The first point is larger or equal to the second
            ii = -1
            if signDx[0] == signDx[1]:  # Want alternating signs
                x = np.concatenate((x[:1], x[2:]))
                ind = np.concatenate((ind[:1], ind[2:]))
                length -= 1

        else:  # First point is smaller than the second
            ii = 0
            if signDx[0] == signDx[1]:  # Want alternating signs
                x = x[1:]
                ind = ind[1:]
                length -= 1

        # Preallocate max number of maxima
        maxPeaks = int(ceil(length / 2.0))
        peak_loc = np.zeros(maxPeaks, dtype=np.int32)
        peak_mag = np.zeros(maxPeaks)
        c_ind = 0
        # Loop through extrema which should be peaks and then valleys
        while ii < (length - 1):
            ii += 1  # This is a peak
            # Reset peak finding if we had a peak and the next peak is bigger
            # than the last or the left min was small enough to reset.
            if found_peak and ((x[ii] > peak_mag[-1]) or
                               (left_min < peak_mag[-1] - thresh)):
                temp_mag = min_mag
                found_peak = False

            # Make sure we don't iterate past the length of our vector
            if ii == length - 1:
                break  # We assign the last point differently out of the loop

            # Found new peak that was lager than temp mag and threshold larger
            # than the minimum to its left.
            if (x[ii] > temp_mag) and (x[ii] > left_min + thresh):
                temp_loc = ii
                temp_mag = x[ii]

            ii += 1  # Move onto the valley
            # Come down at least thresh from peak
            if not found_peak and (temp_mag > (thresh + x[ii])):
                found_peak = True  # We have found a peak
                left_min = x[ii]
                peak_loc[c_ind] = temp_loc  # Add peak to index
                peak_mag[c_ind] = temp_mag
                c_ind += 1
            elif x[ii] < left_min:  # New left minima
                left_min = x[ii]

        # Check end point
        if (x[-1] > temp_mag) and (x[-1] > (left_min + thresh)):
            peak_loc[c_ind] = length - 1
            peak_mag[c_ind] = x[-1]
            c_ind += 1
        elif not found_peak and temp_mag > min_mag:
            # Check if we still need to add the last point
            peak_loc[c_ind] = temp_loc
            peak_mag[c_ind] = temp_mag
            c_ind += 1

        # Create output
        peak_inds = ind[peak_loc[:c_ind]]
        peak_mags = peak_mag[:c_ind]
    else:  # This is a monotone function where an endpoint is the only peak
        x_ind = np.argmax(x)
        peak_mags = x[x_ind]
        if peak_mags > (min_mag + thresh):
            peak_inds = ind[x_ind]
        else:
            peak_mags = []
            peak_inds = []

    # Change sign of data if was finding minima
    if extrema < 0:
        peak_mags *= -1.0
        x0 = -x0

    # Plot if no output desired
    if len(peak_inds) == 0:
        print('No significant peaks found')

    return peak_inds, peak_mags

## Analysis — run this cell **twice** (once per ITD condition)

This is the cell you will run for **each** ITD condition. Here is what it does, step by step:

1. **Find the data.** It looks in the folder `fpath` (e.g. `./S197/`) for all `.bdf` files — the raw BioSemi EEG recordings — and loops over them.
2. **Load and reference.** Each file is read with `mne.io.read_raw_bdf`, and the EEG is re-referenced to the external electrodes `EXG1`/`EXG2` (typically the earlobes/mastoids).
3. **Read the event markers.** `mne.find_events` extracts the trigger codes (1, 3, 5, 7, …) that mark when each ITD jump occurred.
4. **Mark unused channels** (`EXG3`–`EXG8`) as bad so they are ignored during artifact rejection.
5. **Remove blinks with SSP.** On the first file, `find_blinks` detects blinks, short epochs around them are built, and `compute_proj_epochs` computes an SSP projector that captures the blink topography. That projector is then applied to every file. This is the **Signal-Space Projection** method of *Uusitalo & Ilmoniemi (1997)*: it identifies the spatial pattern of the blink and projects it out of the data, leaving the brain response largely intact.
6. **Filter** the continuous data to a 1.5–50 Hz band to remove slow drifts and high-frequency noise.
7. **Epoch and average.** The data are cut into epochs from **−0.5 s to +2.0 s** around each event (the ITD change is at t = 0 within MNE's internal timing, which corresponds to ~1 s in the plotted window), baseline-corrected, and epochs exceeding ±150 µV are rejected as artifacts. Averaging the surviving epochs gives the **evoked response** for the selected condition.
8. **Combine** the per-file evoked responses into a single grand average (`evoked_grand`) and store it in microvolts (`x`) with its time axis (`t`).

### 👉 What you change between the two runs

Inside the cell, find this line:

```python
cond = [1, 5]   # <-- change to [3, 7] for the large ITD jump
```

- **First run — small ITD:** set `cond = [1, 5]`, run this cell, then run the plotting cell below.
- **Second run — large ITD:** change it back to `cond = [3, 7]`, run this cell again, then run the plotting cell again.

Because the plot is built to *add* a new line each time (see below), both conditions will end up overlaid on the same axes for direct comparison.


In [ ]:
fpath = '/content/eegdata/'
bdfs = sorted(fnmatch.filter(os.listdir(fpath), '*.bdf'))

evokeds = []
removeblinks = True

if len(bdfs) >= 1:
    for k, bdf in enumerate(bdfs):
        bdfname = fpath + bdf

        # Load data and read event channel
        raw = mne.io.read_raw_bdf(bdfname, preload=True,
                                  stim_channel='auto')
        raw.set_eeg_reference(ref_channels=['EXG1', 'EXG2'])
        eves = mne.find_events(raw, shortest_event=1, mask=255)

        # Pick channels to not include in epoch rejection
        raw.info['bads'] += ['EXG3', 'EXG4', 'EXG5', 'EXG6', 'EXG7', 'EXG8']


        if removeblinks:
            if k==0:
                # SSP for blinks
                blinks = find_blinks(raw, ch_name=['A1', ])
                epochs_blinks = mne.Epochs(raw, blinks, 998, tmin=-0.25,
                                          tmax=0.25, proj=True,
                                          baseline=(-0.25, 0),
                                          reject=dict(eeg=1000e-6),
                                          verbose='WARNING')
                blink_projs = compute_proj_epochs(epochs_blinks, n_eeg=1,
                                                  verbose='INFO')
                raw.add_proj(blink_projs)
            if k>0:
                raw.add_proj(blink_projs)

        raw.filter(l_freq=1.5, h_freq=50.)


        # Epoch the data
        tmin, tmax = -0.5, 2.0
        baseline=(-0.5, 0.)
        rejthresh = 150e-6

        # >>> STUDENTS: choose which ITD jump to analyze <<<
        # Small ITD jump:  cond = [1, 5]   (1 = L->R leading, 5 = R->L leading)
        # Large ITD jump:  cond = [3, 7]   (3 = L->R leading, 7 = R->L leading)
        # Run this cell once with the small condition, plot it, then
        # change to the large condition, re-run, and plot again.
        cond = [3, 7]   # <-- Set to [1, 5] for the small ITD jump, or [3, 7] for large.
        epochs = mne.Epochs(raw, eves, cond, tmin=tmin, proj=True,
                            tmax=tmax, baseline=baseline,
                            reject=dict(eeg=rejthresh),
                            verbose='INFO')
        evoked = epochs.average()
        evokeds.append(evoked)
else:
    RuntimeError('No BDF files found!!')

evoked_grand = mne.combine_evoked(evokeds, weights='nave')

x = evoked_grand.data * 1e6
t = evoked.times

## Plot the evoked response

The plotting is split into **two cells** on purpose:

- The **setup cell** (next) creates an empty figure **once**.
- The **plotting cell** (after that) draws **one new line** onto that same figure each time you run it.

This is what lets you overlay the small-ITD and large-ITD responses on a single plot: run the setup cell **once**, then run the plotting cell **after each** analysis run.


### Plot setup — run this **once**

This creates an empty figure and axes (`fig`, `ax`) that we will keep adding to. We immediately call `plt.close(fig)` so that running this cell does **not** display a blank, empty plot — the figure object still exists in memory and is reused by the plotting cell below.

⚠️ **Run this cell only once.** If you re-run it, you start a fresh empty figure and lose the lines you have already plotted. (If you ever want to start a clean comparison from scratch, that's exactly when you *would* re-run this cell.)


In [ ]:
# Setup an empty set of axes to plot different conditions on
fig, ax = plt.subplots()
plt.close(fig)

### Draw the response — run this **after each** analysis run

This cell adds the currently-loaded grand-average response as a **new line** on the existing axes, without erasing what's already there. Each time you run the analysis cell for a new condition and then run this cell, another line is overlaid — so after both runs you'll see the small-ITD and large-ITD responses together.

A few details:

- **`chan = 30`** selects the channel to plot (here **Fz**, a fronto-central site where the change complex is typically largest). You can change this index to look at other channels.
- The legend/`N=` in the y-axis label reports how many epochs went into the average.
- The ITD change occurs at about **t = 1 s**. Look for the **N1 (~1.1 s)** and **P2 (~1.2 s)** sensory peaks in the **large** ITD response, and the later **change-detection response (~300 ms after the change, i.e. ~1.3 s)** that appears even for the **small** ITD jump.

> Tip: after plotting both conditions, you may want to add `ax.legend(['Small ITD', 'Large ITD'])` in the order you plotted them to label the two lines.


In [ ]:
# Pick a channel to plot
chan =  30  # Fz
y = x[chan, :]

ax.plot(t, y, linewidth=2)
ax.set_xlabel('Time (s)', fontsize=14)
ax.set_ylabel(rf'Response $(\mu V)$ (N={int(evoked_grand.nave)})', fontsize=14)
ax.grid(True)
ax.tick_params(labelsize=14)
ax.set_xlim(-0.5, 2.)
fig.canvas.draw()
fig          # display without finalizing;